In [11]:
# [code]
# 1. Import Library yang Dibutuhkan
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from supabase import create_client, Client
from dotenv import load_dotenv
import os
from datetime import datetime, timedelta

# Muat variabel lingkungan dari file .env
load_dotenv()

# --- Konfigurasi Supabase ---
# Ganti dengan nama tabel dan kolom timestamp Anda di Supabase
TABLE_NAME = "tb_konsentrasi_gas" 
TIMESTAMP_COLUMN = "created_at" 

# Ambil URL dan Key dari environment variables
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("Pastikan SUPABASE_URL dan SUPABASE_KEY sudah diatur di file .env")

# Inisialisasi klien Supabase
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Library berhasil diimpor dan klien Supabase diinisialisasi.")

Library berhasil diimpor dan klien Supabase diinisialisasi.


In [12]:
# [code]
# 2. Mengambil Data Per Menit (7 Hari Terakhir)

seven_days_ago = datetime.now() - timedelta(days=7)
print(f"Langkah 2: Mengambil data per menit dari {seven_days_ago.strftime('%Y-%m-%d %H:%M')}...")

all_data = []
page_size = 1000
offset = 0

COL_PM25 = "pm25_ugm3"
COL_CO = "co_corrected_ugm3"
COL_SUHU = "temperature"
COL_KELEMBAPAN = "humidity"
COL_TIMESTAMP = "created_at"
TABLE_NAME = "tb_konsentrasi_gas"
while True:
    print(f"Mengambil {page_size} data dimulai dari baris ke-{offset}...")
    response = supabase.table(TABLE_NAME) \
        .select(f"{COL_PM25}, {COL_CO}, {COL_SUHU}, {COL_KELEMBAPAN}, {COL_TIMESTAMP}") \
        .gte(COL_TIMESTAMP, seven_days_ago.isoformat()) \
        .order(COL_TIMESTAMP, desc=False) \
        .range(offset, offset + page_size - 1) \
        .execute()
    
    current_data = response.data
    if not current_data:
        print("Semua data telah diambil.")
        break
        
    all_data.extend(current_data)
    offset += len(current_data)

df_raw = pd.DataFrame(all_data)

if df_raw.empty:
    print("ERROR: Tidak ada data yang ditemukan.")
else:
    print(f"\nData mentah berhasil diambil. Total baris: {len(df_raw)}")
    for col in [COL_PM25, COL_CO, COL_SUHU, COL_KELEMBAPAN]:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
    df_raw[COL_TIMESTAMP] = pd.to_datetime(df_raw[COL_TIMESTAMP])
    initial_count = len(df_raw)
    df_raw.dropna(subset=[COL_PM25, COL_CO, COL_SUHU, COL_KELEMBAPAN], inplace=True)
    final_count = len(df_raw)
    print(f"{initial_count - final_count} baris dengan nilai tidak valid telah dihapus.")
    df_clean = df_raw.set_index(COL_TIMESTAMP)
    df_clean = df_clean[~df_clean.index.duplicated(keep='first')]
    df_clean.sort_index(inplace=True)
    df_clean.rename(columns={
        COL_PM25: 'pm2.5', COL_CO: 'co', COL_SUHU: 'suhu', COL_KELEMBAPAN: 'kelembapan'
    }, inplace=True)
    print("Data per menit yang bersih telah disiapkan.")

Langkah 2: Mengambil data per menit dari 2025-12-05 12:43...
Mengambil 1000 data dimulai dari baris ke-0...


ConnectError: [Errno 11002] getaddrinfo failed

In [ ]:
# [code]
# 3. Rekayasa Fitur (Feature Engineering) untuk Prediksi 1 Menit

if 'df_clean' in locals() and not df_clean.empty:
    print("Langkah 3: Membuat fitur dari data per menit...")
    
    features_df = df_clean.copy()
    feature_cols = ['pm2.5', 'co', 'suhu', 'kelembapan']
    
    # Buat fitur lag dan rolling yang sesuai untuk skala menit
    for col in feature_cols:
        features_df[f'{col}_lag_1min'] = features_df[col].shift(1)
        features_df[f'{col}_lag_5min'] = features_df[col].shift(5)
        features_df[f'{col}_lag_15min'] = features_df[col].shift(15)
        features_df[f'{col}_lag_60min'] = features_df[col].shift(60)
        features_df[f'{col}_rolling_mean_5min'] = features_df[col].rolling(window=5).mean()
        features_df[f'{col}_rolling_std_5min'] = features_df[col].rolling(window=5).std()
        features_df[f'{col}_rolling_mean_15min'] = features_df[col].rolling(window=15).mean()

    # Fitur Berbasis Waktu
    features_df['minute'] = features_df.index.minute
    features_df['hour'] = features_df.index.hour
    features_df['dayofweek'] = features_df.index.dayofweek

    # Tentukan Target: nilai pm2.5 di 1 menit ke depan
    features_df['target'] = features_df['pm2.5'].shift(-1)
    features_df.dropna(inplace=True)

    print(f"\nJumlah baris akhir setelah pembuatan fitur: {len(features_df)}")
    print(f"Jumlah total fitur: {len(features_df.columns)}")
else:
    print("Langkah 3 dilewati.")

Langkah 3: Membuat fitur dari data per menit...

Jumlah baris akhir setelah pembuatan fitur: 9427
Jumlah total fitur: 36


In [ ]:
# [code]
# 4. Persiapan Data untuk Model Machine Learning

if 'features_df' in locals() and not features_df.empty:
    print("Langkah 4: Mempersiapkan data untuk model...")
    
    # Pisahkan fitur (X) dan target (y)
    X = features_df.drop('target', axis=1)
    y = features_df['target']

    # Pembagian data latih dan uji (Time Series Split)
    split_index = int(len(X) * 0.8)
    X_train, X_test = X[:split_index], X[split_index:]
    y_train, y_test = y[:split_index], y[split_index]

    print(f"Ukuran data latih (X_train): {X_train.shape}")
    print(f"Ukuran data uji (X_test): {X_test.shape}")
else:
    print("Langkah 4 dilewati.")

Langkah 4: Mempersiapkan data untuk model...
Ukuran data latih (X_train): (7541, 35)
Ukuran data uji (X_test): (1886, 35)


C:\Users\Yusuf Rajabi\AppData\Local\Temp\ipykernel_22212\2889242587.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  y_train, y_test = y[:split_index], y[split_index]


In [ ]:
!pip install statsmodels
!pip install scikit-learn


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# [code]
# 7. Melatih Model Random Forest

from sklearn.ensemble import RandomForestRegressor

if 'X_train' in locals():
    print("Langkah 7: Melatih model Random Forest...")
    
    model_rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1, verbose=1)
    model_rf.fit(X_train, y_train)
    print("Model Random Forest berhasil dilatih.")
else:
    print("Langkah 7 dilewati.")

Langkah 7: Melatih model Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:   22.4s


Model Random Forest berhasil dilatih.


[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   24.8s finished


In [ ]:
# [code]
# 8. Melatih Model Exponential Smoothing
from sklearn.metrics import r2_score

from statsmodels.tsa.holtwinters import ExponentialSmoothing
if 'df_clean' in locals() and not df_clean.empty:
    print("Langkah 8: Melatih model Exponential Smoothing...")
    
    pm25_series = df_clean['pm2.5'].asfreq('T')
    pm25_series.fillna(method='ffill', inplace=True)

    train_size = int(len(pm25_series) * 0.8)
    train_es = pm25_series.iloc[:train_size]
    test_es = pm25_series.iloc[train_size:]

    model_es = ExponentialSmoothing(train_es, trend='add', seasonal='add', seasonal_periods=1440).fit()
    print("Model Exponential Smoothing selesai dilatih.")
else:
    print("Langkah 8 dilewati.")

Langkah 8: Melatih model Exponential Smoothing...


C:\Users\Yusuf Rajabi\AppData\Local\Temp\ipykernel_22212\511303824.py:9: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  pm25_series = df_clean['pm2.5'].asfreq('T')
C:\Users\Yusuf Rajabi\AppData\Local\Temp\ipykernel_22212\511303824.py:10: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  pm25_series.fillna(method='ffill', inplace=True)


Model Exponential Smoothing selesai dilatih.


c:\Users\Yusuf Rajabi\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


In [ ]:
# [code]
# 9. Evaluasi Semua Model dan Tabel Perbandingan Akhir (VERSI SUPER PERBAIKAN)

# --- Import semua fungsi/metrik yang dibutuhkan di dalam sel ini ---
try:
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    import numpy as np
    import pandas as pd
except ImportError as e:
    print(f"Library tidak ditemukan: {e}. Silakan install.")
    raise

def mean_absolute_percentage_error(y_true, y_pred): 
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred) # Konversi ke array
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# Dictionary untuk menyimpan hasil
results = {}

# --- Evaluasi LightGBM ---
# Bangun kembali data dari sumbernya
if 'model_lgb_final' in locals() and 'features_df' in locals():
    print("Mengevaluasi LightGBM...")
    # Pastikan kita menggunakan data yang sama
    df = features_df.copy()
    split_index = int(len(df) * 0.8)
    X_test = df.drop('target', axis=1).iloc[split_index:]
    y_test = df['target'].iloc[split_index:]
    
    y_pred_lgb = model_lgb_final.predict(X_test)
    
    # Konversi ke array untuk keamanan
    y_true = np.asarray(y_test)
    y_pred = np.asarray(y_pred_lgb)
    
    if y_true.size > 0 and y_pred.size > 0:
        results["LightGBM"] = {
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAPE": mean_absolute_percentage_error(y_true, y_pred),
            "R²": r2_score(y_true, y_pred)
        }

# --- Evaluasi LSTM ---
if 'model_lstm' in locals() and 'features_df' in locals():
    print("Mengevaluasi LSTM...")
    df = features_df.copy()
    split_index = int(len(df) * 0.8)
    X_test_raw = df.drop('target', axis=1).iloc[split_index:]
    y_test_raw = df['target'].iloc[split_index:]
    
    # Buat ulang sequence dan scaling
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_train_full = scaler_X.fit_transform(df.drop('target', axis=1).iloc[:split_index])
    y_train_full = scaler_y.fit_transform(df['target'].iloc[:split_index].values.reshape(-1, 1))
    X_test_scaled = scaler_X.transform(X_test_raw)
    y_test_scaled = scaler_y.transform(y_test_raw.values.reshape(-1, 1))
    
    def create_sequences(X_data, y_data, n_timesteps):
        Xs, ys = [], []
        for i in range(len(X_data) - n_timesteps):
            Xs.append(X_data[i:(i + n_timesteps)])
            ys.append(y_data[i + n_timesteps])
        return np.array(Xs), np.array(ys)
    
    n_timesteps = 60
    _, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, n_timesteps)
    
    y_pred_scaled = model_lstm.predict(X_test_seq)
    y_pred_lstm = scaler_y.inverse_transform(y_pred_scaled)
    y_test_orig = scaler_y.inverse_transform(y_test_seq)
    
    y_true = np.asarray(y_test_orig)
    y_pred = np.asarray(y_pred_lstm)
    
    if y_true.size > 0 and y_pred.size > 0:
        results["LSTM"] = {
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAPE": mean_absolute_percentage_error(y_true, y_pred),
            "R²": r2_score(y_true, y_pred)
        }

# --- Evaluasi Random Forest ---
if 'model_rf' in locals() and 'features_df' in locals():
    print("Mengevaluasi Random Forest...")
    df = features_df.copy()
    split_index = int(len(df) * 0.8)
    X_test = df.drop('target', axis=1).iloc[split_index:]
    y_test = df['target'].iloc[split_index:]
    
    y_pred_rf = model_rf.predict(X_test)
    
    y_true = np.asarray(y_test)
    y_pred = np.asarray(y_pred_rf)

    if y_true.size > 0 and y_pred.size > 0:
        results["Random Forest"] = {
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAPE": mean_absolute_percentage_error(y_true, y_pred),
            "R²": r2_score(y_true, y_pred)
        }

# --- Evaluasi Exponential Smoothing (Dengan Definisi Ulang Total) ---
if 'model_es' in locals() and 'df_clean' in locals():
    print("Mengevaluasi Exponential Smoothing...")
    
    # --- BUAT ULANG SEMUA DATA DARI SUMBER PALING ASLI ---
    pm25_series = df_clean['pm2.5'].asfreq('T')
    pm25_series.fillna(method='ffill', inplace=True)
    train_size = int(len(pm25_series) * 0.8)
    test_es = pm25_series.iloc[train_size:]
    y_pred_es = model_es.forecast(steps=len(test_es))

    # Konversi ke array untuk keamanan
    y_true = np.asarray(test_es)
    y_pred = np.asarray(y_pred_es)
    
    print(f"DEBUG ES: Tipe y_true={type(y_true)}, ukuran={y_true.shape}")
    print(f"DEBUG ES: Tipe y_pred={type(y_pred)}, ukuran={y_pred.shape}")

    if y_true.size > 0 and y_pred.size > 0:
        results["Exp. Smoothing"] = {
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAPE": mean_absolute_percentage_error(y_true, y_pred),
            "R²": r2_score(y_true, y_pred)
        }
    else:
        print("PERINGATAN: Data untuk evaluasi Exponential Smoothing kosong atau tidak valid.")

# Buat DataFrame dari dictionary hasil
if results:
    comparison_df = pd.DataFrame(results).T
    print("\n" + "="*60)
    print(" TABEL PERBANDINGAN AKHIR MODEL PREDIKSI REAL-TIME")
    print("="*60)
    display(comparison_df)
else:
    print("\nTidak ada model yang berhasil dievaluasi. Periksa kembali pelatihan model.")

Mengevaluasi Random Forest...
Mengevaluasi Exponential Smoothing...


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
C:\Users\Yusuf Rajabi\AppData\Local\Temp\ipykernel_22212\2814408032.py:112: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  pm25_series = df_clean['pm2.5'].asfreq('T')
C:\Users\Yusuf Rajabi\AppData\Local\Temp\ipykernel_22212\2814408032.py:113: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  pm25_series.fillna(method='ffill', inplace=True)


DEBUG ES: Tipe y_true=<class 'numpy.ndarray'>, ukuran=(1932,)
DEBUG ES: Tipe y_pred=<class 'numpy.ndarray'>, ukuran=(1932,)

 TABEL PERBANDINGAN AKHIR MODEL PREDIKSI REAL-TIME


,MAE,RMSE,MAPE,R²
Random Forest,9.168244e-01,1.522043e+00,6.770840e+01,0.238435
Exp. Smoothing,4.964582e-14,5.731880e-14,1.611877e-12,-4163.796196
